# Sesi 1 - Prepare Data Knowledge & Create Vector DB

Bagian dari kurikulum **Agentic Knowledge System - FastAPI + DuckDB + Qwen (Local, tanpa LLM cloud)**.

> Jalankan sel dari atas ke bawah secara berurutan. Sel bertanda `🔨 TODO` wajib dilengkapi peserta; solusi contoh disediakan pada sel setelahnya (bisa disembunyikan saat sesi diberikan ke peserta).

## 0. Setup Environment

In [ ]:
!pip -q install duckdb sentence-transformers pypdf

## 1. Konsep: Kenapa Vector Database?
- **Keyword search** cocok untuk pencocokan kata persis.
- **Semantic search** (vector) menangkap *makna*, bukan hanya kata - penting untuk RAG.
- DuckDB dipilih karena **embedded** (satu file `.duckdb`, tanpa server terpisah) dan punya ekstensi `vss` untuk vector similarity search.

In [ ]:
import duckdb

con = duckdb.connect("knowledge.duckdb")
con.execute("INSTALL vss; LOAD vss;")
con.execute("SET hnsw_enable_experimental_persistence=true;")
print(con.execute("PRAGMA database_list").fetchall())

## 2. Membuat Skema Tabel `documents`

In [ ]:
con.execute("""
CREATE TABLE IF NOT EXISTS documents (
    id VARCHAR PRIMARY KEY,
    source VARCHAR,
    content TEXT,
    embedding FLOAT[384],
    created_at TIMESTAMP DEFAULT current_timestamp
);
""")
con.execute("CREATE INDEX IF NOT EXISTS idx_emb ON documents USING HNSW (embedding);")
print("Tabel documents siap.")

## 3. Load Embedding Model
Model `all-MiniLM-L6-v2` menghasilkan vektor 384 dimensi, ringan dan cukup akurat untuk lab ini.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(embed_model.encode("contoh kalimat").shape)

## 4. Contoh Data Mentah (Bahasa Indonesia)
Silakan ganti dengan dokumen milikmu sendiri (FAQ produk, SOP, artikel, dll).

In [ ]:
raw_documents = [
    {"source": "faq_produk.txt", "text": "Garansi produk kami berlaku selama 1 tahun sejak "
        "tanggal pembelian. Klaim garansi dapat dilakukan melalui aplikasi atau toko resmi."},
    {"source": "sop_retur.txt", "text": "Proses retur barang maksimal 7 hari setelah barang "
        "diterima. Barang harus dalam kondisi asli dan menyertakan bukti pembelian."},
    {"source": "faq_pengiriman.txt", "text": "Pengiriman reguler memakan waktu 2-4 hari kerja. "
        "Pengiriman ekspres tersedia untuk kota-kota besar dengan estimasi 1 hari kerja."},
]
len(raw_documents)

## 5. Chunking
Strategi *fixed-size chunking* dengan overlap agar konteks antar-chunk tidak terputus.

In [ ]:
def chunk_text(text, size=60, overlap=15):
    words = text.split()
    chunks = []
    step = max(size - overlap, 1)
    for i in range(0, len(words), step):
        piece = " ".join(words[i:i + size])
        if piece:
            chunks.append(piece)
        if i + size >= len(words):
            break
    return chunks

sample_chunks = chunk_text(raw_documents[0]["text"])
sample_chunks

### 🔨 TODO 1
Ubah `size` dan `overlap` pada `chunk_text`, lalu amati bagaimana jumlah dan isi chunk berubah untuk `raw_documents[1]["text"]`. Tulis pengamatanmu di sel markdown baru.

In [ ]:
# TODO: eksperimen ukuran chunk berbeda
experiment_chunks = chunk_text(raw_documents[1]["text"], size=None, overlap=None)  # isi angka
experiment_chunks

**Solusi contoh:**
```python
experiment_chunks = chunk_text(raw_documents[1]["text"], size=30, overlap=8)
```

## 6. Ingest: Embed & Simpan ke DuckDB

In [ ]:
import uuid

def ingest(text, source):
    inserted = 0
    for chunk in chunk_text(text):
        emb = embed_model.encode(chunk).tolist()
        con.execute(
            "INSERT INTO documents VALUES (?, ?, ?, ?, current_timestamp)",
            [str(uuid.uuid4()), source, chunk, emb]
        )
        inserted += 1
    return inserted

for doc in raw_documents:
    n = ingest(doc["text"], doc["source"])
    print(f"{doc['source']}: {n} chunk disimpan")

In [ ]:
con.execute("SELECT COUNT(*) FROM documents").fetchone()

## 7. Fungsi Semantic Search

In [ ]:
def search(query, k=3):
    q_emb = embed_model.encode(query).tolist()
    rows = con.execute("""
        SELECT source, content, array_distance(embedding, ?::FLOAT[384]) AS dist
        FROM documents ORDER BY dist ASC LIMIT ?
    """, [q_emb, k]).fetchall()
    return [{"source": r[0], "content": r[1], "score": r[2]} for r in rows]

search("berapa lama garansi produk?")

### 🔨 TODO 2
Coba query lain, misalnya tentang retur atau pengiriman. Apakah hasil top-1 selalu relevan? Jika tidak, apa kemungkinan penyebabnya (ukuran chunk, kualitas embedding, jumlah data)?

In [ ]:
# TODO: uji query lain
search("____")

## 8. Ringkasan Sesi 1
- `knowledge.duckdb` sudah berisi dokumen ter-embed dan siap dipakai sebagai sumber data untuk REST API di **Sesi 2**.
- File `knowledge.duckdb` akan dipakai lagi - unduh dari panel Files di Colab bila perlu dipindahkan ke sesi berikutnya.